# Variance Tester

**How consistent is this model, really?** Ask one model the same prompt several times and see how much the answer moves. You get every response side by side, their token cost and latency, and a mechanical consistency read. About a minute, free, no signup.

The thing most people have never checked: ask the same question five times and you can get five different answers. This measures that honestly. It is a consistency read, not a quality score.

A free, own-key tool by [pysolvr / LLM Prover](https://llmprover.pysolvr.com).

---

> **Free tool, provided as-is, no warranty.** It uses *your own* API key and calls the provider directly. **Your key never leaves your machine / your Colab account and is never sent to pysolvr.** You are responsible for your own usage and costs. Outputs are informational, not advice. pysolvr is not liable for decisions made from these results.

**Key safety:** this notebook never asks you to type a key into a cell (see step 2). Keys live in Colab Secrets or your shell environment, so a saved or shared copy contains no secrets.

## 1. Setup

Pulls the tool from the repo and installs the provider SDKs. `openai` also covers Gemini, Grok, Together, and Groq via their OpenAI-compatible endpoints.

In [ ]:
!pip -q install openai>=1.0 anthropic>=0.30
!wget -q -O variance_core.py https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/variance/variance_core.py
!wget -q -O prices.json      https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/variance/prices.json
print('ready')

## 2. Add your API key (do NOT type it into a cell)

**Never paste an API key directly into a notebook cell.** A key typed into a cell gets saved into the notebook file, your Colab history, and any copy you share. Store it *outside* the notebook instead. The cell below only *reads* it.

### On Colab (recommended), use the Secrets panel
1. Click the **key icon** (Secrets) in the left sidebar.
2. **Add a secret** for your model's provider, using the exact name:

   | Secret name | Models it unlocks |
   |-------------|-------------------|
   | `OPENAI_API_KEY` | `gpt-*`, `o3`/`o4-*` |
   | `ANTHROPIC_API_KEY` | `claude*` |
   | `GEMINI_API_KEY` | `gemini*` |
   | `XAI_API_KEY` | `grok*` |
   | `TOGETHER_API_KEY` | `<org>/<model>` |
   | `GROQ_API_KEY` | `qwen/*` |

3. Toggle **Notebook access** on for the secret. You only need the ONE provider whose model you use (this tool runs a single model).

Colab stores these encrypted, per-user, never in the notebook.

### On local Jupyter, environment variable
Set it in your shell *before* launching Jupyter: `export OPENAI_API_KEY=sk-...` then `jupyter lab`.

Then run the cell below. **Nothing is sent to pysolvr; your key stays on your machine / in your Colab account.**

In [ ]:
# Loads keys from Colab Secrets (preferred) or the environment. Contains no key itself.
import os

_KEY_NAMES = ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'XAI_API_KEY', 'TOGETHER_API_KEY', 'GROQ_API_KEY']
try:
    from google.colab import userdata  # Colab only
    for _name in _KEY_NAMES:
        try:
            _val = userdata.get(_name)
            if _val:
                os.environ[_name] = _val
        except Exception:
            pass  # secret not set / notebook access off -- skip
except ImportError:
    pass  # not on Colab -- keys come from the shell environment

_loaded = [n for n in _KEY_NAMES if os.environ.get(n)]
print('keys loaded for:', _loaded if _loaded else 'NONE -- add one via the Colab Secrets panel (key icon, left) with Notebook access ON, or set an env var. See cell above.')

## 3. Pick a model, your prompt, and how many runs

Use the provider's own model-ID string (the tool is model-agnostic). Examples: `gpt-5`, `claude-opus-5-5`, `gemini-4-argon`, `grok-4.7`. To find what exists and what it costs: **[llmprover.pysolvr.com/models](https://llmprover.pysolvr.com/models/)** and **[llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/)**.

- `RUNS`: how many times to fire the same prompt (up to 10). Try a prompt where the model has room to vary (an opinion, a plan, a judgement call), not a lookup with one right answer.
- `TEMPERATURES` (optional): to sweep temperatures instead, set a list like `[0, 0.5, 1.0]`. One run per temperature; this overrides `RUNS`. Leave it as `None` to just repeat at the provider default.

In [ ]:
MODEL  = 'gpt-5'
PROMPT = 'Name the single biggest risk of deploying this model to production, in one sentence.'
RUNS   = 5

# Optional temperature sweep. Set to e.g. [0, 0.5, 1.0] to run one call per temperature.
# Leave as None to repeat RUNS times at the provider default temperature.
TEMPERATURES = None

## 4. Run it

The calls fire concurrently. You get each response, token cost (from the dated price snapshot), and latency, then a mechanical consistency read: the mean, min, and max wording overlap across the responses (1.00 = identical wording, 0.00 = no shared words). **No verdict** -- this is mechanical overlap, not a quality score. You read the spread and judge whether the model is stable enough for your use.

In [ ]:
from variance_core import variance_core, format_result

result = variance_core(
    prompt=PROMPT,
    model=MODEL,
    runs=RUNS,
    temperatures=TEMPERATURES,
)
print(format_result(result))

---

**Prices are a dated snapshot, not a live feed.** For the live, authoritative table see [llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/).

If you're interested in more free tools to test and measure different aspects of an AI pipeline, see our [free AI tools](https://llmprover.pysolvr.com/tools/). For more advanced features, consistency *scored* with a reliability-checked judge, the which-layer diagnostic, and drift tracked over time, see [Features](https://llmprover.pysolvr.com/features/).